# imports

In [44]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn.functional as F
from sklearn.metrics import r2_score
from sklearn.cross_decomposition import CCA
import scipy
from scipy.signal import resample
from scipy.signal import iirnotch, filtfilt, welch, butter
from scipy.spatial.distance import cosine, euclidean
import h5py
from typing import Optional

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, r2_score,confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import pairwise_distances

import seaborn as sns

from utils_correct import create_train_val_loaders, train_ctae_with_logging, safe_format, create_test_loader

# Functions

## data

In [26]:
def load_and_normalize_mat(mat_path):
    with open(mat_path, 'rb') as f:
        header = f.read(128)
    
    is_v73 = b'MATLAB 7.3' in header

    if not is_v73:
        # Use scipy.io for version <= 7.2
        mat = scipy.io.loadmat(mat_path)
        data = mat['matrix_rec']
        fs = float(mat['fs'].squeeze())

    else:
        # Use h5py for v7.3 files
        with h5py.File(mat_path, 'r') as f:
            data = f['matrix_rec'][:].T  # Transpose due to MATLAB column-major format
            fs = float(f['fs'][0][0])


    # Normalize each contact (z-score)
    mean = np.mean(data, axis=1, keepdims=True)
    std = np.std(data, axis=1, keepdims=True)
    data_norm = (data - mean) / (std + 1e-8)

    return data_norm, fs

def downsample(data, original_fs, target_fs):
        num_samples = int(data.shape[1] * target_fs / original_fs)
        return resample(data, num_samples, axis=1)

def segment_data(data, segment_length, fs):
    """
    Args:
        data: (num_contacts, time)
        segment_length: in seconds
        fs: sampling frequency
    Returns:
        list of (num_contacts, segment_samples) arrays
    """
    segment_samples = int(segment_length * fs)
    num_segments = data.shape[1] // segment_samples
    segments = [
        data[:, i*segment_samples : (i+1)*segment_samples]
        for i in range(num_segments)
    ]
    return segments

def load_paired_segments_onstim_wo_vo_with_filtering(folder_path,freq, segment_length=1.0, channel_idx=0, cutoff=50, order=8):
    gpi_data, fs = load_and_normalize_mat(os.path.join(folder_path, f'GPi_{freq}.mat'))
    stn_data, _ = load_and_normalize_mat(os.path.join(folder_path, f'STN_{freq}.mat'))

    target_fs = 500
    gpi_data = downsample(gpi_data, fs, target_fs)
    stn_data = downsample(stn_data, fs, target_fs)

    # Plot PSD before and after filtering for a channel (example: GPi and STN channel 0)
    gpi_filtered = apply_notch_filter(gpi_data, target_fs)
    stn_filtered = apply_notch_filter(stn_data, target_fs)

    # Apply high-order low-pass filter
    gpi_filtered = apply_lowpass_filter(gpi_filtered, target_fs, cutoff, order)
    stn_filtered = apply_lowpass_filter(stn_filtered, target_fs, cutoff, order)

    # plot_psd_comparison(gpi_data, gpi_filtered, target_fs, channel_idx)
    # plot_psd_comparison(stn_data, stn_filtered, target_fs, channel_idx)
    # plot_time_comparison(gpi_data, gpi_filtered, target_fs, channel_idx)
    # plot_time_comparison(stn_data, stn_filtered, target_fs, channel_idx)


    gpi_segments = segment_data(gpi_filtered, segment_length, target_fs)
    stn_segments = segment_data(stn_filtered, segment_length, target_fs)

    print("segments of gpi", np.shape(gpi_segments))
    print("segments of stn", np.shape(stn_segments))

    num_pairs = min(len(gpi_segments), len(stn_segments))
    gpi_segments = gpi_segments[:num_pairs]
    stn_segments = stn_segments[:num_pairs]

    return gpi_segments, stn_segments, target_fs

def build_dataset_with_lag_lagged_stn_wo_vo(gpi_segs, stn_segs, lags=3): 
    """
    Creates lagged input and output segments from GPi and STN.

    Returns:
        X: (segments, in_channels * (lags+1), time - lags)
        y: (segments, out_channels * (lags+1), time - lags)
    """
    X, y = [], []
    for gpi, stn in zip(gpi_segs, stn_segs):
        # --- Input: GPi ---
        lagged_input = [gpi[:, i:gpi.shape[1] - lags + i] for i in range(lags + 1)]
        X_lag = np.concatenate(lagged_input, axis=0)  # (in_channels * (lags+1), time - lags)

        # --- Output: STN (also lagged) ---
        lagged_stn = [stn[:, i:stn.shape[1] - lags + i] for i in range(lags + 1)]
        y_lag = np.concatenate(lagged_stn, axis=0)  # (out_channels * (lags+1), time - lags)

        X.append(X_lag)
        y.append(y_lag)

    return np.stack(X), np.stack(y)  # (segments, in_channels, time), (segments, out_channels, time)

def apply_notch_filter(data, fs, freqs=[60], Q=40):
    #assuming input has shape channels, time
    data_t = np.transpose(data, (1, 0))  # [time, channels]
    filtered = data_t.copy()

    for f0 in freqs:
        b, a = iirnotch(f0, Q, fs)
        for ch in range(filtered.shape[1]):
            filtered[:, ch] = filtfilt(b, a, filtered[:, ch])

    return np.transpose(filtered, (1, 0))  # back to [channels, time]

def apply_lowpass_filter(data, fs, cutoff=50, order=8):
    """
    Apply a lowpass Butterworth filter to multi-channel time series data.
    
    Args:
        data (numpy array): shape (channels, time)
        fs (float): sampling frequency
        cutoff (float): cutoff frequency in Hz
        order (int): filter order

    Returns:
        filtered_data (numpy array): same shape as input
    """
    b, a = butter(order, cutoff / (fs / 2), btype='low')
    filtered = data.copy()
    for ch in range(data.shape[0]):
        filtered[ch, :] = filtfilt(b, a, data[ch, :])
    return filtered


## offstim eval

In [7]:

def _to_np(x):
    return x.detach().cpu().numpy()


def _safe_cca_topk(z1, z2, k=None, eps=1e-8):
    """
    z1, z2: torch tensors [1, T, D] or [T, D]
    Returns mean top-k canonical correlation.
    """
    z1 = _to_np(z1)
    z2 = _to_np(z2)

    z1 = np.squeeze(z1)
    z2 = np.squeeze(z2)

    if z1.ndim != 2 or z2.ndim != 2:
        return None

    # shape should be [T, D]
    if z1.shape[0] < 3 or z2.shape[0] < 3:
        return None

    d = min(z1.shape[1], z2.shape[1])
    if d < 1:
        return None

    if k is None:
        k = d
    else:
        k = min(k, d)

    # avoid zero-variance crashes
    if np.any(np.std(z1, axis=0) < eps) or np.any(np.std(z2, axis=0) < eps):
        return None

    try:
        cca = CCA(n_components=k, max_iter=1000)
        u, v = cca.fit_transform(z1, z2)

        corrs = []
        for j in range(k):
            if np.std(u[:, j]) < eps or np.std(v[:, j]) < eps:
                continue
            corrs.append(np.corrcoef(u[:, j], v[:, j])[0, 1])

        if len(corrs) == 0:
            return None

        return float(np.mean(np.abs(corrs)))

    except Exception:
        return None


def get_ctae_measuresAll_df_per_sample(
    model,
    gpi_test,
    stn_test,
    device,
    side,
    subject_id=None,
    num_neurons1=None,
):
    """
    CTAE equivalent of get_measuresAll_df_per_sample_v2.

    Inputs:
        gpi_test: [N, T, Cgpi]
        stn_test: [N, T, Cstn]

    Returns:
        df_recon:
            Per-sample reconstruction MSE/R2 for:
              - full own-region reconstruction
              - shared-only cross reconstruction

        df_sim:
            Per-sample CCA similarity/leakage metrics:
              - shared_gpi ~ shared_stn
              - shared_gpi ~ private_gpi
              - shared_stn ~ private_stn
              - private_gpi ~ private_stn

        df_ratio:
            Per-sample cross-shared transfer ratios.
    """

    model = model.to(device)
    model.eval()

    recon_records = []
    sim_records = []
    ratio_records = []

    N = min(gpi_test.shape[0], stn_test.shape[0])
    T = min(gpi_test.shape[1], stn_test.shape[1])

    gpi_test = gpi_test[:N, :T, :]
    stn_test = stn_test[:N, :T, :]

    if num_neurons1 is None:
        num_neurons1 = gpi_test.shape[-1]

    for i in range(N):
        xb = gpi_test[i:i + 1].to(device).float()
        yb = stn_test[i:i + 1].to(device).float()

        # CTAE expects concatenated input: [B, T, Cgpi + Cstn]
        data_cur = torch.cat([xb, yb], dim=-1)

        with torch.no_grad():
            (
                x11_hat,
                x22_hat,
                x12_hat,
                x21_hat,
                shared_gpi,
                shared_stn,
                private_gpi,
                private_stn,
                z,
            ) = model(data_cur, num_neurons1=num_neurons1)

        # CTAE decoder outputs are [T, B, C], convert to [B, T, C]
        gpi_full = x11_hat.permute(1, 0, 2)
        stn_full = x22_hat.permute(1, 0, 2)

        # shared-only cross reconstructions
        stn_shared_gpi = x12_hat.permute(1, 0, 2)  # STN reconstructed from shared
        gpi_shared_stn = x21_hat.permute(1, 0, 2)  # GPi reconstructed from shared

        outputs = {
            "gpi_full": gpi_full,
            "stn_full": stn_full,
            "gpi_shared_cross": gpi_shared_stn,
            "stn_shared_cross": stn_shared_gpi,
        }

        targets = {
            "gpi": xb,
            "stn": yb,
        }

        mse_cache = {}

        for key, out in outputs.items():
            region = key.split("_")[0]
            tgt = targets[region]

            mse = F.mse_loss(out, tgt, reduction="mean").item()
            r2 = r2_score(_to_np(tgt).ravel(), _to_np(out).ravel())

            mse_cache[key] = mse

            recon_records.append({
                "subject": subject_id,
                "side": side,
                "sample": i,
                "target_region": region,
                "condition": key,
                "mse": mse,
                "r2": r2,
                "model": "CTAE",
            })

        eps = 1e-12

        # CTAE does not provide same-region shared-only recon directly,
        # so this ratio is full-vs-cross shared rather than same-shared-vs-cross-shared.
        ratio_records.append({
            "subject": subject_id,
            "side": side,
            "sample": i,
            "target_region": "gpi",
            "metric": "cross_shared_vs_full_mse_ratio",
            "numerator_condition": "gpi_shared_cross",
            "denominator_condition": "gpi_full",
            "value": mse_cache["gpi_shared_cross"] / (mse_cache["gpi_full"] + eps),
            "model": "CTAE",
        })

        ratio_records.append({
            "subject": subject_id,
            "side": side,
            "sample": i,
            "target_region": "stn",
            "metric": "cross_shared_vs_full_mse_ratio",
            "numerator_condition": "stn_shared_cross",
            "denominator_condition": "stn_full",
            "value": mse_cache["stn_shared_cross"] / (mse_cache["stn_full"] + eps),
            "model": "CTAE",
        })

        # -------------------------
        # CCA similarity / leakage
        # -------------------------
        cca_shared = _safe_cca_topk(shared_gpi, shared_stn, k=None)
        cca_leak_gpi = _safe_cca_topk(shared_gpi, private_gpi, k=None)
        cca_leak_stn = _safe_cca_topk(shared_stn, private_stn, k=None)
        cca_private_cross = _safe_cca_topk(private_gpi, private_stn, k=None)

        cca_items = [
            ("shared_gpi ~ shared_stn", cca_shared),
            ("shared_gpi ~ private_gpi", cca_leak_gpi),
            ("shared_stn ~ private_stn", cca_leak_stn),
            ("private_gpi ~ private_stn", cca_private_cross),
        ]

        for pair, value in cca_items:
            if value is not None:
                sim_records.append({
                    "subject": subject_id,
                    "side": side,
                    "sample": i,
                    "pair": pair,
                    "metric": "cca_mean",
                    "value": value,
                    "model": "CTAE",
                })

    df_recon = pd.DataFrame(recon_records)
    df_sim = pd.DataFrame(sim_records)
    df_ratio = pd.DataFrame(ratio_records)

    return df_recon, df_sim, df_ratio

In [8]:
def compute_lfp_zscore_stats(x, eps=1e-8):
    # x: [N, T, C]
    mean = x.reshape(-1, x.shape[-1]).mean(dim=0)
    std = x.reshape(-1, x.shape[-1]).std(dim=0)
    return mean, std + eps


def apply_lfp_zscore(x, mean, std):
    return (x - mean) / std


def prepare_ctae_pair_from_tensors(gpi, stn, gpi_mean=None, gpi_std=None, stn_mean=None, stn_std=None):
    """
    Returns:
        gpi_z, stn_z, data, stats
    """
    gpi = gpi.float()
    stn = stn.float()

    N = min(gpi.shape[0], stn.shape[0])
    T = min(gpi.shape[1], stn.shape[1])

    gpi = gpi[:N, :T, :]
    stn = stn[:N, :T, :]

    if gpi_mean is None or gpi_std is None:
        gpi_mean, gpi_std = compute_lfp_zscore_stats(gpi)

    if stn_mean is None or stn_std is None:
        stn_mean, stn_std = compute_lfp_zscore_stats(stn)

    gpi_z = apply_lfp_zscore(gpi, gpi_mean, gpi_std)
    stn_z = apply_lfp_zscore(stn, stn_mean, stn_std)

    data = torch.cat([gpi_z, stn_z], dim=-1)  # [N, T, Cgpi+Cstn]

    stats = {
        "gpi_mean": gpi_mean,
        "gpi_std": gpi_std,
        "stn_mean": stn_mean,
        "stn_std": stn_std,
    }

    return gpi_z, stn_z, data, stats

In [45]:

def compute_latent_psd(
    latent_td: np.ndarray,
    fs: float,
    nperseg: Optional[int] = None,
    noverlap: Optional[int] = None,
    detrend: str = "constant",
):
    """
    latent_td: (T, D)
    Returns:
        freqs: (F,)
        psd:   (D, F)
    """
    latent_td = np.asarray(latent_td)
    if latent_td.ndim != 2:
        raise ValueError(f"Expected latent_td with shape (T, D), got {latent_td.shape}")

    T, D = latent_td.shape
    if nperseg is None:
        nperseg = min(128, T)
    if noverlap is None:
        noverlap = nperseg // 2

    psd_all = []
    freqs = None
    for d in range(D):
        f, pxx = welch(
            latent_td[:, d],
            fs=fs,
            nperseg=nperseg,
            noverlap=noverlap,
            detrend=detrend,
            scaling="density",
        )
        if freqs is None:
            freqs = f
        psd_all.append(pxx)

    psd_all = np.stack(psd_all, axis=0)  # (D, F)
    return freqs, psd_all

def spectral_centroid_from_psd(freqs: np.ndarray, psd: np.ndarray, fmin=None, fmax=None, eps: float = 1e-12):
    """
    freqs: (F,)
    psd:   (D, F) or (F,)
    Returns centroid per dim if psd is 2D, otherwise scalar
    """
    freqs = np.asarray(freqs)
    psd = np.asarray(psd)

    idx = np.ones_like(freqs, dtype=bool)
    if fmin is not None:
        idx &= (freqs >= fmin)
    if fmax is not None:
        idx &= (freqs <= fmax)

    f_sel = freqs[idx]
    p_sel = psd[..., idx]

    num = np.sum(p_sel * f_sel, axis=-1)
    den = np.sum(p_sel, axis=-1) + eps
    return num / den


def extract_ctae_test_latents(
    model,
    gpi_test,
    stn_test,
    device,
    num_neurons1=None,
    batch_size=16,
):
    """
    Returns CTAE latents as numpy arrays [N, T, D].
    """

    model = model.to(device)
    model.eval()

    if num_neurons1 is None:
        num_neurons1 = gpi_test.shape[-1]

    N = min(gpi_test.shape[0], stn_test.shape[0])
    T = min(gpi_test.shape[1], stn_test.shape[1])

    gpi_test = gpi_test[:N, :T, :].float()
    stn_test = stn_test[:N, :T, :].float()

    shared_gpi_all = []
    shared_stn_all = []
    private_gpi_all = []
    private_stn_all = []

    with torch.no_grad():
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)

            xb = gpi_test[start:end].to(device)
            yb = stn_test[start:end].to(device)

            data_cur = torch.cat([xb, yb], dim=-1)

            (
                x11_hat,
                x22_hat,
                x12_hat,
                x21_hat,
                shared_gpi,
                shared_stn,
                private_gpi,
                private_stn,
                z,
            ) = model(data_cur, num_neurons1=num_neurons1)

            # CTAE returns [T, B, D], convert to [B, T, D]
            shared_gpi_all.append(shared_gpi.permute(1, 0, 2).detach().cpu())
            shared_stn_all.append(shared_stn.permute(1, 0, 2).detach().cpu())
            private_gpi_all.append(private_gpi.permute(1, 0, 2).detach().cpu())
            private_stn_all.append(private_stn.permute(1, 0, 2).detach().cpu())

    latents = {
        "shared_gpi": torch.cat(shared_gpi_all, dim=0).numpy(),
        "shared_stn": torch.cat(shared_stn_all, dim=0).numpy(),
        "private_gpi": torch.cat(private_gpi_all, dim=0).numpy(),
        "private_stn": torch.cat(private_stn_all, dim=0).numpy(),
    }

    return latents


def analyze_ctae_latent_spectral_centroid(
    model,
    gpi_test,
    stn_test,
    device,
    fs=500.0,
    subject_id=None,
    side=None,
    nperseg=None,
    fmin=1.0,
    fmax=50.0,
    batch_size=16,
):
    """
    CTAE version.
    Returns:
        df_centroid, latents
    """

    latents = extract_ctae_test_latents(
        model=model,
        gpi_test=gpi_test,
        stn_test=stn_test,
        device=device,
        num_neurons1=gpi_test.shape[-1],
        batch_size=batch_size,
    )

    records = []

    for latent_name, arr_ntd in latents.items():
        N = arr_ntd.shape[0]

        for i in range(N):
            x_td = arr_ntd[i]  # [T, D]

            freqs, psd = compute_latent_psd(
                x_td,
                fs=fs,
                nperseg=nperseg,
            )

            centroids = spectral_centroid_from_psd(
                freqs,
                psd,
                fmin=fmin,
                fmax=fmax,
            )

            for d, c in enumerate(np.asarray(centroids)):
                records.append({
                    "model": "CTAE",
                    "subject": subject_id,
                    "side": side,
                    "sample": i,
                    "latent_type": latent_name,
                    "dim": d,
                    "spectral_centroid_hz": float(c),
                })

    df_centroid = pd.DataFrame(records)

    return df_centroid, latents

## onstim eval

In [29]:
def extract_ctae_latents_by_condition(
    model,
    gpi_all,
    stn_all,
    labels_all,
    device,
    label_map,
    num_neurons1=None,
    batch_size=16,
):
    model = model.to(device)
    model.eval()

    if num_neurons1 is None:
        num_neurons1 = gpi_all.shape[-1]  # GPi channels only

    N = min(gpi_all.shape[0], stn_all.shape[0])
    T = min(gpi_all.shape[1], stn_all.shape[1])

    gpi_all = gpi_all[:N, :T, :].float()
    stn_all = stn_all[:N, :T, :].float()
    labels_all = labels_all[:N]

    shared_gpi = {name: [] for name in label_map.values()}
    shared_stn = {name: [] for name in label_map.values()}
    private_gpi = {name: [] for name in label_map.values()}
    private_stn = {name: [] for name in label_map.values()}

    with torch.no_grad():
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)

            xb = gpi_all[start:end].to(device)
            yb = stn_all[start:end].to(device)
            lb = labels_all[start:end].cpu()

            data_cur = torch.cat([xb, yb], dim=-1)

            (
                x11_hat,
                x22_hat,
                x12_hat,
                x21_hat,
                shared_subspace1,
                shared_subspace2,
                specific_subspace1,
                specific_subspace2,
                z,
            ) = model(data_cur, num_neurons1=num_neurons1)

            # CTAE outputs latents as [T, B, D], convert to [B, T, D]
            shared_subspace1 = shared_subspace1.permute(1, 0, 2).detach().cpu()
            shared_subspace2 = shared_subspace2.permute(1, 0, 2).detach().cpu()
            specific_subspace1 = specific_subspace1.permute(1, 0, 2).detach().cpu()
            specific_subspace2 = specific_subspace2.permute(1, 0, 2).detach().cpu()

            for local_i, label in enumerate(lb):
                condition_name = label_map[int(label.item())]

                shared_gpi[condition_name].append(shared_subspace1[local_i])
                shared_stn[condition_name].append(shared_subspace2[local_i])
                private_gpi[condition_name].append(specific_subspace1[local_i])
                private_stn[condition_name].append(specific_subspace2[local_i])

    def stack_dict(d):
        out = {}
        for key, vals in d.items():
            if len(vals) > 0:
                out[key] = torch.stack(vals, dim=0)  # [Ncond, T, D]
            else:
                out[key] = torch.empty(0)
        return out

    return (
        stack_dict(shared_gpi),
        stack_dict(shared_stn),
        stack_dict(private_gpi),
        stack_dict(private_stn),
    )

In [32]:
def prepare_latents_without_averaging(latent_dict):
    data = []
    labels = []
    condition_map = {"Off": 0, "85Hz": 1, "185Hz": 2, "250Hz": 3}

    for condition, tensor in latent_dict.items():
        if condition not in condition_map:
            continue

        tensor = tensor.detach().cpu().numpy() if torch.is_tensor(tensor) else tensor

        if tensor.size == 0:
            continue

        N, T, D = tensor.shape
        flat = tensor.reshape(N * T, D)

        data.append(flat)
        labels.extend([condition_map[condition]] * (N * T))

    X = np.vstack(data)
    y = np.array(labels)

    return X, y

def calculate_RF_accuracy(shared_gpi, shared_stn, private_gpi, private_stn, side, setting, subject_id):
    latent_sets = {
        "shared_gpi": shared_gpi,
        "shared_stn": shared_stn,
        "private_gpi": private_gpi,
        "private_stn": private_stn,
    }

    results_accuracy = []
    results_importance = []

    for latent_type, latent_dict in latent_sets.items():
        X, y = prepare_latents_without_averaging(latent_dict)

        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, stratify=y, test_size=0.2, random_state=42
        )

        # Normalize
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # Classifier
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        acc = accuracy_score(y_test, y_pred)

        # Save classification accuracy
        results_accuracy.append({
            "subject": subject_id,
            "side": side,
            "setting": setting,
            "latent_type": latent_type,
            "accuracy": acc,
            "precision_macro": precision_score(y_test, y_pred, average='macro'),
            "recall_macro": recall_score(y_test, y_pred, average='macro'),
            "f1_macro": f1_score(y_test, y_pred, average='macro'),
            "n_samples": len(y)
        })

        # Save feature importances
        for i, imp in enumerate(clf.feature_importances_):
            results_importance.append({
                "subject": subject_id,
                "side": side,
                "setting": setting,
                "latent_type": latent_type,
                "feature_dim": i,
                "importance": imp,
            })

    return pd.DataFrame(results_accuracy), pd.DataFrame(results_importance)

In [40]:
def calculate_centroid_shift(shared_gpi, shared_stn, private_gpi, private_stn, side, setting, subject_id):
    """
    Calculates centroid shift (cosine and euclidean) between off-stim and other frequencies
    for each latent type. Assumes latents are dictionaries with keys like "Off", "85Hz", etc.
    Each value should be of shape (N, T, dim).
    Returns a DataFrame with relevant shift data.
    how the average latent representation changes under stimulation
    """
    def to_np(x):
        return x.detach().cpu().numpy() if torch.is_tensor(x) else x

    all_latents = {
        "shared_gpi": shared_gpi,
        "shared_stn": shared_stn,
        "private_gpi": private_gpi,
        "private_stn": private_stn,
    }

    rows = []
    for latent_type, latent_dict in all_latents.items():
        # off_mean = latent_dict["Off"].mean(dim=(0, 1)).numpy()
        off = to_np(latent_dict["Off"])
        off_mean = off.mean(axis=(0, 1))

        for freq in ["85Hz", "185Hz", "250Hz"]:
            if freq not in latent_dict:
                continue
            # freq_mean = latent_dict[freq].mean(dim=(0, 1)).numpy()
            freq_latent = to_np(latent_dict[freq])
            freq_mean = freq_latent.mean(axis=(0, 1))

            cos_shift = cosine(off_mean, freq_mean)
            euc_shift = euclidean(off_mean, freq_mean)

            rows.append({
                "subject": subject_id,
                "side": side,
                "setting": setting,
                "frequency": freq,
                "latent_type": latent_type,
                "cosine_shift": cos_shift,
                "euclidean_shift": euc_shift,
            })

    return pd.DataFrame(rows)

# Usage

## offstim eval

In [9]:

# Your data are LFP windows, not binned spikes
fs = 500          # after downsampling
window_sec = 0.5
bin_size = 1 / fs # only used if you need a time vector

# Transformer settings: start small
nhead = 1
num_layers = 2

learning_rate = 1e-4

# Start conservative
lambda_ortho = 1e-3
lambda_alignment = 0.05
lambda_recons2 = 1
lambda_shared = 1

warm_up_ortho = 20

# Positional encoding
pe = True
pe_learn = False

# Your windows are ~247 or 250 timepoints
max_len = 300

batch_size = 8   # or 16 if GPU memory allows
num_epochs = 300 # start with 300, not 5000

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


cuda:0
NVIDIA RTX 5000 Ada Generation


### MSE

In [13]:
subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side
# subject_list = ["s508","s514", "s515","s517","s519","s520","s521","s523"]  # right side

excel_save_dir = r"D:\copy_comp_project\ctae_models\excels" #make sure directory exists
DATA_ROOT = r"D:\copy_comp_project\Off_tensor_Data_L"#################
TRAINED_MODELS_ROOT = r"D:\copy_comp_project\ctae_models"

side="L"
# target_epoch = 140  # or 199

# neurips_plus_align_recover_F_R with help of var: 508:3,2 514:3,3 ,515:3,4 ,517:3,2 ,519: 5,2 ,520:3,4 ,521:5,2 ,523!:5,4
subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}
if side == "R":
    subject_dims = subject_dims_R
elif side=="L":
    subject_dims = subject_dims_L

rand_init_seed = 702  
LAGS= 3

all_dfs_recon = []  # collect all subject results
all_dfs_simil = []  # collect all subject results
all_dfs_ratio = []

for subj in subject_list:
    print(f"\n Processing subject {subj}...")
    SUBJ_DIR = os.path.join(DATA_ROOT, subj)
    # Latent dimensions: match your SPIRE grid
    shared_latent_dim, r1_specific_dim = subject_dims[subj]
    r2_specific_dim = r1_specific_dim      # STN private

    # ------------------------
    # 1. Load tensors
    # Expected shape from your SPIRE pipeline: [N, W, C]
    # N = windows, W = timepoints, C = channels
    # ------------------------
    gpi_train = torch.load(os.path.join(SUBJ_DIR, "gpi_train_off.pt"), map_location="cpu").float()
    stn_train = torch.load(os.path.join(SUBJ_DIR, "stn_train_off.pt"), map_location="cpu").float()
    gpi_test_raw = torch.load(os.path.join(SUBJ_DIR, "gpi_test_off.pt"))
    stn_test_raw = torch.load(os.path.join(SUBJ_DIR, "stn_test_off.pt"))

    print(f"Loaded: {gpi_test_raw.shape}, {stn_test_raw.shape}")

    #2. prep tensors
    _ , _ , _ , norm_stats = prepare_ctae_pair_from_tensors(
        gpi_train,
        stn_train,
    )

    gpi_test, stn_test, data_test_tensor, _ = prepare_ctae_pair_from_tensors(
        gpi_test_raw,
        stn_test_raw,
        gpi_mean=norm_stats["gpi_mean"].cpu(),
        gpi_std=norm_stats["gpi_std"].cpu(),
        stn_mean=norm_stats["stn_mean"].cpu(),
        stn_std=norm_stats["stn_std"].cpu(),
    )

    # 3. load model
    model_path = (
            f"{TRAINED_MODELS_ROOT}/ctae_{subj}_{side}"
            f"_bs{batch_size}"
            f"_lr{safe_format(learning_rate)}"
            f"_L{num_layers}"
            f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
            f"_s{shared_latent_dim}"
            f"_pe{'T' if pe else 'F'}"
            f"_align{safe_format(lambda_alignment)}"
            f"_ortho{safe_format(lambda_ortho)}"
            f"_recons2-{safe_format(lambda_recons2)}"
            f"_warm{warm_up_ortho}"
            f"_seed{rand_init_seed}"
            f"_ep{num_epochs}.pth"
        )
    print(f"\n=== {model_path} ===")
    bundle_path = model_path.replace(".pth", "_bundle.pth")
    bundle = torch.load(bundle_path, map_location=device, weights_only=False)

    model = bundle["model"]
    model = model.to(device)
    model.eval()

    print(f"model Loaded")

    # 4. Run and collect metrics
    df_recon_ctae, df_sim_ctae, df_ratio_ctae = get_ctae_measuresAll_df_per_sample(
        model=model,
        gpi_test=gpi_test,
        stn_test=stn_test,
        device=device,
        side=side,
        subject_id=subj,
        num_neurons1=gpi_test.shape[-1],
    )

    all_dfs_recon.append(df_recon_ctae)
    all_dfs_simil.append(df_sim_ctae)
    all_dfs_ratio.append(df_ratio_ctae)

# Concatenate all and save
final_df_recon = pd.concat(all_dfs_recon, ignore_index=True)
final_df_recon.to_excel(os.path.join(excel_save_dir, f"MSE_results_CTAE_{side}.xlsx"), index=False)####L
final_df_simil = pd.concat(all_dfs_simil, ignore_index=True)
final_df_simil.to_excel(os.path.join(excel_save_dir, f"simil_results_CTAE_{side}.xlsx"), index=False)####L, _sd{sd}_pd{pdim}
# final_df_ratio = pd.concat(all_dfs_ratio, ignore_index=True)
# final_df_ratio.to_excel(
#     os.path.join(excel_save_dir, f"ratio_results_{run_prefix}_{side}_v2.xlsx"),
#     index=False
# )
print("✅ Excel file saved!")


 Processing subject s508...
Loaded: torch.Size([121, 247, 72]), torch.Size([121, 247, 12])

=== D:\copy_comp_project\ctae_models/ctae_s508_L_bs8_lr0.0001_L2_r1-4_r2-4_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s513...
Loaded: torch.Size([455, 247, 24]), torch.Size([455, 247, 24])

=== D:\copy_comp_project\ctae_models/ctae_s513_L_bs8_lr0.0001_L2_r1-3_r2-3_s5_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s514...
Loaded: torch.Size([164, 247, 44]), torch.Size([164, 247, 16])

=== D:\copy_comp_project\ctae_models/ctae_s514_L_bs8_lr0.0001_L2_r1-2_r2-2_s5_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s515...
Loaded: torch.Size([576, 247, 24]), torch.Size([576, 247, 12])

=== D:\copy_comp_project\ctae_models/ctae_s515_L_bs8_lr0.0001_L2_r1-4_r2-4_s5_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Proces

ModuleNotFoundError: No module named 'openpyxl'

In [15]:
# subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side
subject_list = ["s508","s514", "s515","s517","s519","s520","s521","s523"]  # right side

excel_save_dir = r"D:\copy_comp_project\ctae_models\excels" #make sure directory exists
DATA_ROOT = r"D:\copy_comp_project\Off_tensor_Data_R"#################
TRAINED_MODELS_ROOT = r"D:\copy_comp_project\ctae_models"

side="R"
# target_epoch = 140  # or 199

# neurips_plus_align_recover_F_R with help of var: 508:3,2 514:3,3 ,515:3,4 ,517:3,2 ,519: 5,2 ,520:3,4 ,521:5,2 ,523!:5,4
subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}
if side == "R":
    subject_dims = subject_dims_R
elif side=="L":
    subject_dims = subject_dims_L

rand_init_seed = 702  
LAGS= 3

all_dfs_recon = []  # collect all subject results
all_dfs_simil = []  # collect all subject results
all_dfs_ratio = []

for subj in subject_list:
    print(f"\n Processing subject {subj}...")
    SUBJ_DIR = os.path.join(DATA_ROOT, subj)
    # Latent dimensions: match your SPIRE grid
    shared_latent_dim, r1_specific_dim = subject_dims[subj]
    r2_specific_dim = r1_specific_dim      # STN private

    # ------------------------
    # 1. Load tensors
    # Expected shape from your SPIRE pipeline: [N, W, C]
    # N = windows, W = timepoints, C = channels
    # ------------------------
    gpi_train = torch.load(os.path.join(SUBJ_DIR, "gpi_train_off.pt"), map_location="cpu").float()
    stn_train = torch.load(os.path.join(SUBJ_DIR, "stn_train_off.pt"), map_location="cpu").float()
    gpi_test_raw = torch.load(os.path.join(SUBJ_DIR, "gpi_test_off.pt"))
    stn_test_raw = torch.load(os.path.join(SUBJ_DIR, "stn_test_off.pt"))

    print(f"Loaded: {gpi_test_raw.shape}, {stn_test_raw.shape}")

    #2. prep tensors
    _ , _ , _ , norm_stats = prepare_ctae_pair_from_tensors(
        gpi_train,
        stn_train,
    )

    gpi_test, stn_test, data_test_tensor, _ = prepare_ctae_pair_from_tensors(
        gpi_test_raw,
        stn_test_raw,
        gpi_mean=norm_stats["gpi_mean"].cpu(),
        gpi_std=norm_stats["gpi_std"].cpu(),
        stn_mean=norm_stats["stn_mean"].cpu(),
        stn_std=norm_stats["stn_std"].cpu(),
    )

    # 3. load model
    model_path = (
            f"{TRAINED_MODELS_ROOT}/ctae_{subj}_{side}"
            f"_bs{batch_size}"
            f"_lr{safe_format(learning_rate)}"
            f"_L{num_layers}"
            f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
            f"_s{shared_latent_dim}"
            f"_pe{'T' if pe else 'F'}"
            f"_align{safe_format(lambda_alignment)}"
            f"_ortho{safe_format(lambda_ortho)}"
            f"_recons2-{safe_format(lambda_recons2)}"
            f"_warm{warm_up_ortho}"
            f"_seed{rand_init_seed}"
            f"_ep{num_epochs}.pth"
        )
    print(f"\n=== {model_path} ===")
    bundle_path = model_path.replace(".pth", "_bundle.pth")
    bundle = torch.load(bundle_path, map_location=device, weights_only=False)

    model = bundle["model"]
    model = model.to(device)
    model.eval()

    print(f"model Loaded")

    # 4. Run and collect metrics
    df_recon_ctae, df_sim_ctae, df_ratio_ctae = get_ctae_measuresAll_df_per_sample(
        model=model,
        gpi_test=gpi_test,
        stn_test=stn_test,
        device=device,
        side=side,
        subject_id=subj,
        num_neurons1=gpi_test.shape[-1],
    )

    all_dfs_recon.append(df_recon_ctae)
    all_dfs_simil.append(df_sim_ctae)
    all_dfs_ratio.append(df_ratio_ctae)

# Concatenate all and save
final_df_recon = pd.concat(all_dfs_recon, ignore_index=True)
final_df_recon.to_excel(os.path.join(excel_save_dir, f"MSE_results_CTAE_{side}.xlsx"), index=False)####L
final_df_simil = pd.concat(all_dfs_simil, ignore_index=True)
final_df_simil.to_excel(os.path.join(excel_save_dir, f"simil_results_CTAE_{side}.xlsx"), index=False)####L, _sd{sd}_pd{pdim}
# final_df_ratio = pd.concat(all_dfs_ratio, ignore_index=True)
# final_df_ratio.to_excel(
#     os.path.join(excel_save_dir, f"ratio_results_{run_prefix}_{side}_v2.xlsx"),
#     index=False
# )
print("✅ Excel file saved!")


 Processing subject s508...
Loaded: torch.Size([121, 247, 48]), torch.Size([121, 247, 24])

=== D:\copy_comp_project\ctae_models/ctae_s508_R_bs8_lr0.0001_L2_r1-2_r2-2_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded


c:\Users\rahil\Documents\ctae-sanger\venv_ctae2\Lib\site-packages\sklearn\cross_decomposition\_pls.py:113: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)



 Processing subject s514...
Loaded: torch.Size([164, 247, 32]), torch.Size([164, 247, 16])

=== D:\copy_comp_project\ctae_models/ctae_s514_R_bs8_lr0.0001_L2_r1-3_r2-3_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s515...
Loaded: torch.Size([576, 247, 12]), torch.Size([576, 247, 12])

=== D:\copy_comp_project\ctae_models/ctae_s515_R_bs8_lr0.0001_L2_r1-4_r2-4_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s517...
Loaded: torch.Size([144, 247, 12]), torch.Size([144, 247, 16])

=== D:\copy_comp_project\ctae_models/ctae_s517_R_bs8_lr0.0001_L2_r1-2_r2-2_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s519...
Loaded: torch.Size([137, 247, 32]), torch.Size([137, 247, 16])

=== D:\copy_comp_project\ctae_models/ctae_s519_R_bs8_lr0.0001_L2_r1-2_r2-2_s5_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Proces

### freq analysis

In [48]:
subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side
# subject_list = ["s508","s514", "s515","s517","s519","s520","s521","s523"]  # right side

excel_save_dir = r"D:\copy_comp_project\ctae_models\excels" #make sure directory exists
DATA_ROOT = r"D:\copy_comp_project\Off_tensor_Data_L"#################
TRAINED_MODELS_ROOT = r"D:\copy_comp_project\ctae_models"

side="L"
# target_epoch = 140  # or 199

# neurips_plus_align_recover_F_R with help of var: 508:3,2 514:3,3 ,515:3,4 ,517:3,2 ,519: 5,2 ,520:3,4 ,521:5,2 ,523!:5,4
subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}
if side == "R":
    subject_dims = subject_dims_R
elif side=="L":
    subject_dims = subject_dims_L

rand_init_seed = 702  
LAGS= 3

all_centroid = []  # collect all subject results

for subj in subject_list:
    print(f"\n Processing subject {subj}...")
    SUBJ_DIR = os.path.join(DATA_ROOT, subj)
    # Latent dimensions: match your SPIRE grid
    shared_latent_dim, r1_specific_dim = subject_dims[subj]
    r2_specific_dim = r1_specific_dim      # STN private

    # ------------------------
    # 1. Load tensors
    # Expected shape from your SPIRE pipeline: [N, W, C]
    # N = windows, W = timepoints, C = channels
    # ------------------------
    gpi_train = torch.load(os.path.join(SUBJ_DIR, "gpi_train_off.pt"), map_location="cpu").float()
    stn_train = torch.load(os.path.join(SUBJ_DIR, "stn_train_off.pt"), map_location="cpu").float()
    gpi_test_raw = torch.load(os.path.join(SUBJ_DIR, "gpi_test_off.pt"))
    stn_test_raw = torch.load(os.path.join(SUBJ_DIR, "stn_test_off.pt"))

    print(f"Loaded: {gpi_test_raw.shape}, {stn_test_raw.shape}")

    #2. prep tensors
    _ , _ , _ , norm_stats = prepare_ctae_pair_from_tensors(
        gpi_train,
        stn_train,
    )

    gpi_test, stn_test, data_test_tensor, _ = prepare_ctae_pair_from_tensors(
        gpi_test_raw,
        stn_test_raw,
        gpi_mean=norm_stats["gpi_mean"].cpu(),
        gpi_std=norm_stats["gpi_std"].cpu(),
        stn_mean=norm_stats["stn_mean"].cpu(),
        stn_std=norm_stats["stn_std"].cpu(),
    )

    # 3. load model
    model_path = (
            f"{TRAINED_MODELS_ROOT}/ctae_{subj}_{side}"
            f"_bs{batch_size}"
            f"_lr{safe_format(learning_rate)}"
            f"_L{num_layers}"
            f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
            f"_s{shared_latent_dim}"
            f"_pe{'T' if pe else 'F'}"
            f"_align{safe_format(lambda_alignment)}"
            f"_ortho{safe_format(lambda_ortho)}"
            f"_recons2-{safe_format(lambda_recons2)}"
            f"_warm{warm_up_ortho}"
            f"_seed{rand_init_seed}"
            f"_ep{num_epochs}.pth"
        )
    print(f"\n=== {model_path} ===")
    bundle_path = model_path.replace(".pth", "_bundle.pth")
    bundle = torch.load(bundle_path, map_location=device, weights_only=False)

    model = bundle["model"]
    model = model.to(device)
    model.eval()

    print(f"model Loaded")

    # 4. Run and collect metrics
    df_centroid, _ = analyze_ctae_latent_spectral_centroid(
        model=model,
        gpi_test=gpi_test,
        stn_test=stn_test,
        device=device,
        fs=500,
        side=side,
        subject_id=subj,
        fmin=1.0,
        fmax=50.0,
        batch_size=16,
    )

    all_centroid.append(df_centroid)


# Concatenate all and save
final_df_centroid = pd.concat(all_centroid, ignore_index=True)
final_df_centroid.to_excel(os.path.join(excel_save_dir, f"spectral_centroid_results_CTAE_{side}.xlsx"), index=False)####L

print("✅ Excel file saved!")


 Processing subject s508...
Loaded: torch.Size([121, 247, 72]), torch.Size([121, 247, 12])

=== D:\copy_comp_project\ctae_models/ctae_s508_L_bs8_lr0.0001_L2_r1-4_r2-4_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s513...
Loaded: torch.Size([455, 247, 24]), torch.Size([455, 247, 24])

=== D:\copy_comp_project\ctae_models/ctae_s513_L_bs8_lr0.0001_L2_r1-3_r2-3_s5_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s514...
Loaded: torch.Size([164, 247, 44]), torch.Size([164, 247, 16])

=== D:\copy_comp_project\ctae_models/ctae_s514_L_bs8_lr0.0001_L2_r1-2_r2-2_s5_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s515...
Loaded: torch.Size([576, 247, 24]), torch.Size([576, 247, 12])

=== D:\copy_comp_project\ctae_models/ctae_s515_L_bs8_lr0.0001_L2_r1-4_r2-4_s5_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Proces

In [49]:
# subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side
subject_list = ["s508","s514", "s515","s517","s519","s520","s521","s523"]  # right side

excel_save_dir = r"D:\copy_comp_project\ctae_models\excels" #make sure directory exists
DATA_ROOT = r"D:\copy_comp_project\Off_tensor_Data_R"#################
TRAINED_MODELS_ROOT = r"D:\copy_comp_project\ctae_models"

side="R"
# target_epoch = 140  # or 199

# neurips_plus_align_recover_F_R with help of var: 508:3,2 514:3,3 ,515:3,4 ,517:3,2 ,519: 5,2 ,520:3,4 ,521:5,2 ,523!:5,4
subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}
if side == "R":
    subject_dims = subject_dims_R
elif side=="L":
    subject_dims = subject_dims_L

rand_init_seed = 702  
LAGS= 3

all_centroid = []  # collect all subject results

for subj in subject_list:
    print(f"\n Processing subject {subj}...")
    SUBJ_DIR = os.path.join(DATA_ROOT, subj)
    # Latent dimensions: match your SPIRE grid
    shared_latent_dim, r1_specific_dim = subject_dims[subj]
    r2_specific_dim = r1_specific_dim      # STN private

    # ------------------------
    # 1. Load tensors
    # Expected shape from your SPIRE pipeline: [N, W, C]
    # N = windows, W = timepoints, C = channels
    # ------------------------
    gpi_train = torch.load(os.path.join(SUBJ_DIR, "gpi_train_off.pt"), map_location="cpu").float()
    stn_train = torch.load(os.path.join(SUBJ_DIR, "stn_train_off.pt"), map_location="cpu").float()
    gpi_test_raw = torch.load(os.path.join(SUBJ_DIR, "gpi_test_off.pt"))
    stn_test_raw = torch.load(os.path.join(SUBJ_DIR, "stn_test_off.pt"))

    print(f"Loaded: {gpi_test_raw.shape}, {stn_test_raw.shape}")

    #2. prep tensors
    _ , _ , _ , norm_stats = prepare_ctae_pair_from_tensors(
        gpi_train,
        stn_train,
    )

    gpi_test, stn_test, data_test_tensor, _ = prepare_ctae_pair_from_tensors(
        gpi_test_raw,
        stn_test_raw,
        gpi_mean=norm_stats["gpi_mean"].cpu(),
        gpi_std=norm_stats["gpi_std"].cpu(),
        stn_mean=norm_stats["stn_mean"].cpu(),
        stn_std=norm_stats["stn_std"].cpu(),
    )

    # 3. load model
    model_path = (
            f"{TRAINED_MODELS_ROOT}/ctae_{subj}_{side}"
            f"_bs{batch_size}"
            f"_lr{safe_format(learning_rate)}"
            f"_L{num_layers}"
            f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
            f"_s{shared_latent_dim}"
            f"_pe{'T' if pe else 'F'}"
            f"_align{safe_format(lambda_alignment)}"
            f"_ortho{safe_format(lambda_ortho)}"
            f"_recons2-{safe_format(lambda_recons2)}"
            f"_warm{warm_up_ortho}"
            f"_seed{rand_init_seed}"
            f"_ep{num_epochs}.pth"
        )
    print(f"\n=== {model_path} ===")
    bundle_path = model_path.replace(".pth", "_bundle.pth")
    bundle = torch.load(bundle_path, map_location=device, weights_only=False)

    model = bundle["model"]
    model = model.to(device)
    model.eval()

    print(f"model Loaded")

    # 4. Run and collect metrics
    df_centroid, _ = analyze_ctae_latent_spectral_centroid(
        model=model,
        gpi_test=gpi_test,
        stn_test=stn_test,
        device=device,
        fs=500,
        side=side,
        subject_id=subj,
        fmin=1.0,
        fmax=50.0,
        batch_size=16,
    )

    all_centroid.append(df_centroid)


# Concatenate all and save
final_df_centroid = pd.concat(all_centroid, ignore_index=True)
final_df_centroid.to_excel(os.path.join(excel_save_dir, f"spectral_centroid_results_CTAE_{side}.xlsx"), index=False)####L

print("✅ Excel file saved!")


 Processing subject s508...
Loaded: torch.Size([121, 247, 48]), torch.Size([121, 247, 24])

=== D:\copy_comp_project\ctae_models/ctae_s508_R_bs8_lr0.0001_L2_r1-2_r2-2_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s514...
Loaded: torch.Size([164, 247, 32]), torch.Size([164, 247, 16])

=== D:\copy_comp_project\ctae_models/ctae_s514_R_bs8_lr0.0001_L2_r1-3_r2-3_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s515...
Loaded: torch.Size([576, 247, 12]), torch.Size([576, 247, 12])

=== D:\copy_comp_project\ctae_models/ctae_s515_R_bs8_lr0.0001_L2_r1-4_r2-4_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Processing subject s517...
Loaded: torch.Size([144, 247, 12]), torch.Size([144, 247, 16])

=== D:\copy_comp_project\ctae_models/ctae_s517_R_bs8_lr0.0001_L2_r1-2_r2-2_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded

 Proces

## onstim

### save test tensors

In [31]:
# subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side
# subject_list = ["s508","s514", "s515","s517","s519","s520","s521","s523"]  # right side

base_dir = r"D:\copy_comp_project\LPF_Data\imagingContacts"  # Base directory path
TRAINED_MODELS_ROOT = r"D:\copy_comp_project\ctae_models"
save_test_latent_dir = r"D:\copy_comp_project\ctae_models\test_latents"

# subject_list = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
subject_list = ["s519"]
# target_epoch = 140  # or 199

# neurips_plus_align_recover_F_R with help of var: 508:3,2 514:3,3 ,515:3,4 ,517:3,2 ,519: 5,2 ,520:3,4 ,521:5,2 ,523!:5,4
subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}


rand_init_seed = 702  
LAGS= 3


for subj in subject_list:
    print(f"\n Processing subject {subj}...")
    onstim_path_subj = os.path.join(base_dir, subj)
    # Filter for folders that start with "GPi"
    gpi_folders = [
        f for f in os.listdir(onstim_path_subj)
        if os.path.isdir(os.path.join(onstim_path_subj, f)) and f.startswith("GPi")
    ]

    for setting in gpi_folders:
        path_onstim = os.path.join(onstim_path_subj, setting)
        
        # Extract the side (first character after the underscore)
        try:
            side = setting.split("_")[1][0]  # 'L' from 'L12'
        except IndexError:
            side = "?"  # fallback if the format is unexpected
        
        # --- pick dims by subject & side ---
        if side == "R":
            dims_map = subject_dims_R
            off_dir = r"D:\copy_comp_project\Off_tensor_Data_R" #####side
        elif side == "L":
            dims_map = subject_dims_L
            off_dir = r"D:\copy_comp_project\Off_tensor_Data_L" #####side
        else:
            print(f"[WARN] Unknown side '{side}' in setting '{setting}'. Skipping.")
            continue
        if subj not in dims_map:
            print(f"[WARN] No dims configured for subject {subj} on side {side}. Skipping.")
            continue

        SUBJ_DIR = os.path.join(off_dir, subj)
        # Latent dimensions: match your SPIRE grid
        shared_latent_dim, r1_specific_dim = dims_map[subj]
        r2_specific_dim = r1_specific_dim      # STN private

        # ------------------------
        # 1. Load offstim tensors
        # Expected shape from your SPIRE pipeline: [N, W, C]
        # N = windows, W = timepoints, C = channels
        # ------------------------
        gpi_train = torch.load(os.path.join(SUBJ_DIR, "gpi_train_off.pt"), map_location="cpu").float()
        stn_train = torch.load(os.path.join(SUBJ_DIR, "stn_train_off.pt"), map_location="cpu").float()
        gpi_test_raw = torch.load(os.path.join(SUBJ_DIR, "gpi_test_off.pt"))
        stn_test_raw = torch.load(os.path.join(SUBJ_DIR, "stn_test_off.pt"))

        print(f"Loaded: {gpi_test_raw.shape}, {stn_test_raw.shape}")
        n_off = 35

        # Randomly sample n_off segments from X_test_off
        indices = torch.randperm(gpi_test_raw.size(0))[:n_off]
        X_test_off_sub = gpi_test_raw[indices]
        y_test_off_sub = stn_test_raw[indices]

        # make onstim tensors
        freq = 85  # or 185 or 250
        gpi_segs_on, stn_segs_on, fs_on = load_paired_segments_onstim_wo_vo_with_filtering(path_onstim, freq, segment_length=0.5,channel_idx=0, cutoff=50,order=11)
        X_on, y_on = build_dataset_with_lag_lagged_stn_wo_vo(gpi_segs_on, stn_segs_on, lags=3)
        X_tensor_85 = torch.tensor(X_on, dtype=torch.float32).permute(0, 2, 1)
        y_tensor_85 = torch.tensor(y_on, dtype=torch.float32).permute(0, 2, 1)

        freq = 185  # or 185 or 250
        gpi_segs_on, stn_segs_on, fs_on = load_paired_segments_onstim_wo_vo_with_filtering(path_onstim, freq, segment_length=0.5,channel_idx=0, cutoff=50,order=11)
        X_on, y_on = build_dataset_with_lag_lagged_stn_wo_vo(gpi_segs_on, stn_segs_on, lags=3)
        X_tensor_185 = torch.tensor(X_on, dtype=torch.float32).permute(0, 2, 1)
        y_tensor_185 = torch.tensor(y_on, dtype=torch.float32).permute(0, 2, 1)

        freq = 250  # or 185 or 250
        gpi_segs_on, stn_segs_on, fs_on = load_paired_segments_onstim_wo_vo_with_filtering(path_onstim, freq, segment_length=0.5,channel_idx=0, cutoff=50,order=11)
        X_on, y_on = build_dataset_with_lag_lagged_stn_wo_vo(gpi_segs_on, stn_segs_on, lags=3)
        X_tensor_250 = torch.tensor(X_on, dtype=torch.float32).permute(0, 2, 1) #N, T, ch
        y_tensor_250 = torch.tensor(y_on, dtype=torch.float32).permute(0, 2, 1)

        

        #2. prep tensors
        _ , _ , _ , norm_stats = prepare_ctae_pair_from_tensors(
            gpi_train,
            stn_train,
        )
        gpi_mean=norm_stats["gpi_mean"].cpu()
        gpi_std=norm_stats["gpi_std"].cpu()
        stn_mean=norm_stats["stn_mean"].cpu()
        stn_std=norm_stats["stn_std"].cpu()

        X_test_off_sub_z, y_test_off_sub_z, _ , _ = prepare_ctae_pair_from_tensors(
            X_test_off_sub, y_test_off_sub,
            gpi_mean, gpi_std,
            stn_mean, stn_std,
        )

        X_85_z, y_85_z, _ , _ = prepare_ctae_pair_from_tensors(
            X_tensor_85, y_tensor_85,
            gpi_mean, gpi_std,
            stn_mean, stn_std,
        )

        X_185_z, y_185_z, _ , _ = prepare_ctae_pair_from_tensors(
            X_tensor_185, y_tensor_185,
            gpi_mean, gpi_std,
            stn_mean, stn_std,
        )

        X_250_z, y_250_z, _ , _ = prepare_ctae_pair_from_tensors(
            X_tensor_250, y_tensor_250,
            gpi_mean, gpi_std,
            stn_mean, stn_std,
        )

        X_test_all = torch.cat([X_test_off_sub_z, X_85_z, X_185_z, X_250_z], dim=0)
        Y_test_all = torch.cat([y_test_off_sub_z, y_85_z, y_185_z, y_250_z], dim=0)

        labels_test_all = torch.cat([
            torch.zeros(len(X_test_off_sub_z)),
            torch.ones(len(X_85_z)),
            2 * torch.ones(len(X_185_z)),
            3 * torch.ones(len(X_250_z)),
        ]).long()

        label_map = {0: "Off", 1: "85Hz", 2: "185Hz", 3: "250Hz"}

        # 3. load model
        model_path = (
                f"{TRAINED_MODELS_ROOT}/ctae_{subj}_{side}"
                f"_bs{batch_size}"
                f"_lr{safe_format(learning_rate)}"
                f"_L{num_layers}"
                f"_r1-{r1_specific_dim}_r2-{r2_specific_dim}"
                f"_s{shared_latent_dim}"
                f"_pe{'T' if pe else 'F'}"
                f"_align{safe_format(lambda_alignment)}"
                f"_ortho{safe_format(lambda_ortho)}"
                f"_recons2-{safe_format(lambda_recons2)}"
                f"_warm{warm_up_ortho}"
                f"_seed{rand_init_seed}"
                f"_ep{num_epochs}.pth"
            )
        print(f"\n=== {model_path} ===")
        bundle_path = model_path.replace(".pth", "_bundle.pth")
        bundle = torch.load(bundle_path, map_location=device, weights_only=False)

        model = bundle["model"]
        model = model.to(device)
        model.eval()

        print(f"model Loaded")

        ## finally extract test latents for each condition
        shared_gpi, shared_stn, private_gpi, private_stn = extract_ctae_latents_by_condition(
            model=model,
            gpi_all=X_test_all,
            stn_all=Y_test_all,
            labels_all=labels_test_all,
            device=device,
            label_map=label_map,
            num_neurons1=X_test_all.shape[-1],
            batch_size=16,
        )

        save_dir = os.path.join(save_test_latent_dir, subj, setting)
        os.makedirs(save_dir, exist_ok=True)

        torch.save(shared_gpi, os.path.join(save_dir, "shared_gpi.pt"))
        torch.save(shared_stn, os.path.join(save_dir, "shared_stn.pt"))
        torch.save(private_gpi, os.path.join(save_dir, "private_gpi.pt"))
        torch.save(private_stn, os.path.join(save_dir, "private_stn.pt"))

    




 Processing subject s519...
Loaded: torch.Size([267, 247, 36]), torch.Size([267, 247, 12])
segments of gpi (30, 9, 250)
segments of stn (30, 3, 250)
segments of gpi (39, 9, 250)
segments of stn (39, 3, 250)
segments of gpi (33, 9, 250)
segments of stn (33, 3, 250)

=== D:\copy_comp_project\ctae_models/ctae_s519_L_bs8_lr0.0001_L2_r1-4_r2-4_s3_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded
Loaded: torch.Size([137, 247, 32]), torch.Size([137, 247, 16])
segments of gpi (30, 8, 250)
segments of stn (30, 4, 250)
segments of gpi (39, 8, 250)
segments of stn (39, 4, 250)
segments of gpi (33, 8, 250)
segments of stn (33, 4, 250)

=== D:\copy_comp_project\ctae_models/ctae_s519_R_bs8_lr0.0001_L2_r1-2_r2-2_s5_peT_align0.05_ortho0.001_recons2-1_warm20_seed702_ep300.pth ===
model Loaded
Loaded: torch.Size([267, 247, 36]), torch.Size([267, 247, 12])
segments of gpi (30, 9, 250)
segments of stn (30, 3, 250)
segments of gpi (39, 9, 250)
segments of stn (39, 3, 250)
segmen

### classification

In [34]:
# subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side
# subject_list = ["s508","s514", "s515","s517","s519","s520","s521","s523"]  # right side


save_test_latent_dir = r"D:\copy_comp_project\ctae_models\test_latents"
excel_save_dir = r"D:\copy_comp_project\ctae_models\excels"

subject_list = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
# subject_list = ["s519"]
# target_epoch = 140  # or 199

# neurips_plus_align_recover_F_R with help of var: 508:3,2 514:3,3 ,515:3,4 ,517:3,2 ,519: 5,2 ,520:3,4 ,521:5,2 ,523!:5,4
subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}


rand_init_seed = 702  
LAGS= 3
all_acc_dfs, all_imp_dfs = [], []

for subj in subject_list:
    print(f"\n Processing subject {subj}...")
    onstim_path_subj = os.path.join(save_test_latent_dir, subj)
    # Filter for folders that start with "GPi"
    gpi_folders = [
        f for f in os.listdir(onstim_path_subj)
        if os.path.isdir(os.path.join(onstim_path_subj, f)) and f.startswith("GPi")
    ]

    for setting in gpi_folders:
        path_latents = os.path.join(onstim_path_subj, setting)
        
        # Extract the side (first character after the underscore)
        try:
            side = setting.split("_")[1][0]  # 'L' from 'L12'
        except IndexError:
            side = "?"  # fallback if the format is unexpected


        label_map = {0: "Off", 1: "85Hz", 2: "185Hz", 3: "250Hz"}

        #load latents
        label_map = {0: "Off", 1: "85Hz", 2: "185Hz", 3: "250Hz"}
        shared_gpi = torch.load(os.path.join(path_latents, "shared_gpi.pt"))
        shared_stn = torch.load(os.path.join(path_latents, "shared_stn.pt"))
        private_gpi = torch.load(os.path.join(path_latents, "private_gpi.pt"))
        private_stn = torch.load(os.path.join(path_latents, "private_stn.pt"))

        print(f"Test latents Loaded")

        #calculate the measure for pointwise ditribution shift for each latent type
        df_acc, df_imp = calculate_RF_accuracy(shared_gpi, shared_stn, private_gpi, private_stn, side, setting, subj)
        all_acc_dfs.append(df_acc)
        all_imp_dfs.append(df_imp)

# Concatenate all and save
final_acc_df = pd.concat(all_acc_dfs, ignore_index=True)
# final_imp_df = pd.concat(all_imp_dfs, ignore_index=True)
final_acc_df.to_excel(os.path.join(excel_save_dir, "RF_accuracy_CTAE.xlsx"), index=False)
# final_imp_df.to_excel(os.path.join(excel_save_dir, "RF_importance_F.xlsx"), index=False)

print("✅ Excel file saved!")
    




 Processing subject s508...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s513...
Test latents Loaded
Test latents Loaded

 Processing subject s514...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s515...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s517...
Test latents Loaded

 Processing subject s518...
Test latents Loaded
Test latents Loaded

 Processing subject s519...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s520...
Test latents Loaded
Test latents Loaded

 Processing subject s521...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s523...
Test latents Loaded
Test latents Loaded
Test laten

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\copy_comp_project\\ctae_models\\test_latents\\s519_3region'

In [35]:
# Concatenate all and save
final_acc_df = pd.concat(all_acc_dfs, ignore_index=True)
# final_imp_df = pd.concat(all_imp_dfs, ignore_index=True)
final_acc_df.to_excel(os.path.join(excel_save_dir, "RF_accuracy_CTAE.xlsx"), index=False)
# final_imp_df.to_excel(os.path.join(excel_save_dir, "RF_importance_F.xlsx"), index=False)

print("✅ Excel file saved!")

✅ Excel file saved!


### centroid shift

In [41]:
# subject_list = ["s508", "s513","s514", "s515","s518","s519","s520","s521","s523"]  # left side
# subject_list = ["s508","s514", "s515","s517","s519","s520","s521","s523"]  # right side


save_test_latent_dir = r"D:\copy_comp_project\ctae_models\test_latents"
excel_save_dir = r"D:\copy_comp_project\ctae_models\excels"

subject_list = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
# subject_list = ["s519"]
# target_epoch = 140  # or 199

# neurips_plus_align_recover_F_R with help of var: 508:3,2 514:3,3 ,515:3,4 ,517:3,2 ,519: 5,2 ,520:3,4 ,521:5,2 ,523!:5,4
subject_dims_R = {  # subject-specific (sd, pdim)
    "s508": (3,2),
    "s514": (3,3),
    "s515": (3,4),
    "s517": (3,2),
    "s519": (5,2),
    "s520": (3,4),
    "s521": (5, 2),
    "s523": (5,4),
}
# neurips_plus_align_recover_F_L with help of var: 508:3,4/ 513:5,3/ 514:5,2/ 515: 5,4/ 518: 3,2/ 519:3,4/ 520:4,3/521: 5,2 /523: 5,4
subject_dims_L = {  # subject-specific (sd, pdim)
    "s508": (3,4),
    "s513": (5,3),
    "s514": (5,2),
    "s515": (5,4),
    "s518": (3,2),
    "s519": (3,4),
    "s520": (4,3),
    "s521": (5, 2),
    "s523": (5,4),
}


rand_init_seed = 702  
LAGS= 3
all_shift = []

for subj in subject_list:
    print(f"\n Processing subject {subj}...")
    onstim_path_subj = os.path.join(save_test_latent_dir, subj)
    # Filter for folders that start with "GPi"
    gpi_folders = [
        f for f in os.listdir(onstim_path_subj)
        if os.path.isdir(os.path.join(onstim_path_subj, f)) and f.startswith("GPi")
    ]

    for setting in gpi_folders:
        path_latents = os.path.join(onstim_path_subj, setting)
        
        # Extract the side (first character after the underscore)
        try:
            side = setting.split("_")[1][0]  # 'L' from 'L12'
        except IndexError:
            side = "?"  # fallback if the format is unexpected

        #load latents
        label_map = {0: "Off", 1: "85Hz", 2: "185Hz", 3: "250Hz"}
        shared_gpi = torch.load(os.path.join(path_latents, "shared_gpi.pt"))
        shared_stn = torch.load(os.path.join(path_latents, "shared_stn.pt"))
        private_gpi = torch.load(os.path.join(path_latents, "private_gpi.pt"))
        private_stn = torch.load(os.path.join(path_latents, "private_stn.pt"))

        print(f"Test latents Loaded")

        #calculate the measure for pointwise ditribution shift for each latent type
        df_shift = calculate_centroid_shift(shared_gpi, shared_stn, private_gpi, private_stn, side, setting, subj)
        all_shift.append(df_shift)

# Concatenate all and save
final_shift_df = pd.concat(all_shift, ignore_index=True)
final_shift_df.to_excel(os.path.join(excel_save_dir, "shift_stim_CTAE.xlsx"), index=False)

print("✅ Excel file saved!")
    




 Processing subject s508...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s513...
Test latents Loaded
Test latents Loaded

 Processing subject s514...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s515...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s517...
Test latents Loaded

 Processing subject s518...
Test latents Loaded
Test latents Loaded

 Processing subject s519...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s520...
Test latents Loaded
Test latents Loaded

 Processing subject s521...
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded
Test latents Loaded

 Processing subject s523...
Test latents Loaded
Test latents Loaded
Test laten

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\copy_comp_project\\ctae_models\\test_latents\\s519_3region'

In [42]:
# Concatenate all and save
final_shift_df = pd.concat(all_shift, ignore_index=True)
final_shift_df.to_excel(os.path.join(excel_save_dir, "shift_stim_CTAE.xlsx"), index=False)

print("✅ Excel file saved!")

✅ Excel file saved!
